# Chapter 6 · Attention and the Transformer

A small character-level GPT, trained on *Don Quixote*. It uses PyTorch on the CPU and needs no GPU. Run the cells in order; training takes about six minutes.

The full explanation is in the PDF chapter. This notebook is the lab: the same program, split into parts.

## 1. Attention by hand

Check the chapter's example: three positions with their queries, keys and values. The mask puts −∞ in the future and softmax turns the scores into weights.

In [1]:
import torch
queries = torch.tensor([[0., 1.], [0., 1.], [1., 0.]])
keys = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
values = torch.tensor([[2., 0.], [0., 2.], [1., 1.]])
scores = queries @ keys.T / 2 ** 0.5
mask = torch.tril(torch.ones(3, 3))
scores = scores.masked_fill(mask == 0, float("-inf"))
weights = torch.softmax(scores, dim=-1)
print("Weights:\n", weights.round(decimals=3))
print("Outputs:\n", (weights @ values).round(decimals=3))

Weights:
 tensor([[1.0000, 0.0000, 0.0000],
        [0.3300, 0.6700, 0.0000],
        [0.4010, 0.1980, 0.4010]])
Outputs:
 tensor([[2.0000, 0.0000],
        [0.6600, 1.3400],
        [1.2030, 0.7970]])


## 2. The data and the batches

In Colab, the first cell downloads the novel if it is not there.

In [2]:
"""Chapter 6: a small GPT trained on Don Quixote. PyTorch on CPU."""
import math
import time
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

TEXT = Path("material/quixote.txt")
if not TEXT.exists():  # in Colab, download it from the repository
    import urllib.request
    TEXT.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/HernanDiaz/language-models/main/material/quixote.txt", TEXT)
CONTEXT, DIMENSION, HEADS, LAYERS = 64, 64, 4, 4
BATCH, STEPS, RATE, SEED = 32, 3000, 3e-3, 1337


def prepare(text):
    symbols = sorted(set(text))
    to_id = {c: i for i, c in enumerate(symbols)}
    data = torch.tensor([to_id[c] for c in text], dtype=torch.long)
    cut = int(0.9 * len(data))
    return symbols, to_id, data[:cut], data[cut:]


def get_batch(data, generator):
    limit = len(data) - CONTEXT - 1
    starts = torch.randint(limit, (BATCH,), generator=generator)
    x = torch.stack([data[i:i + CONTEXT] for i in starts])
    y = torch.stack([data[i + 1:i + CONTEXT + 1] for i in starts])
    return x, y

## 3. A head, a block and the model

In [3]:
class Head(nn.Module):
    """A causal attention head."""

    def __init__(self, size):
        super().__init__()
        self.query = nn.Linear(DIMENSION, size, bias=False)
        self.key = nn.Linear(DIMENSION, size, bias=False)
        self.value = nn.Linear(DIMENSION, size, bias=False)
        mask = torch.tril(torch.ones(CONTEXT, CONTEXT))
        self.register_buffer("mask", mask)

    def forward(self, x):
        T = x.shape[1]
        q, k, v = self.query(x), self.key(x), self.value(x)
        scores = q @ k.transpose(1, 2) / math.sqrt(k.shape[-1])
        future = self.mask[:T, :T] == 0
        scores = scores.masked_fill(future, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        self.last_weights = weights.detach()  # so we can look at them later
        return weights @ v

In [4]:
class Block(nn.Module):
    """Attention across positions, then an MLP at each position."""

    def __init__(self):
        super().__init__()
        size = DIMENSION // HEADS
        self.heads = nn.ModuleList([Head(size) for _ in range(HEADS)])
        self.projection = nn.Linear(DIMENSION, DIMENSION)
        self.mlp = nn.Sequential(nn.Linear(DIMENSION, 4 * DIMENSION), nn.ReLU(),
                                 nn.Linear(4 * DIMENSION, DIMENSION))
        self.norm1 = nn.LayerNorm(DIMENSION)
        self.norm2 = nn.LayerNorm(DIMENSION)

    def forward(self, x):
        h = self.norm1(x)
        attention = torch.cat([head(h) for head in self.heads], dim=-1)
        x = x + self.projection(attention)
        x = x + self.mlp(self.norm2(x))
        return x

In [5]:
class GPT(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.symbol_embedding = nn.Embedding(vocab_size, DIMENSION)
        self.position_embedding = nn.Embedding(CONTEXT, DIMENSION)
        self.blocks = nn.Sequential(*[Block() for _ in range(LAYERS)])
        self.norm = nn.LayerNorm(DIMENSION)
        self.output = nn.Linear(DIMENSION, vocab_size)

    def forward(self, ids):
        T = ids.shape[1]
        positions = torch.arange(T)
        x = self.symbol_embedding(ids) + self.position_embedding(positions)
        return self.output(self.norm(self.blocks(x)))

In [6]:
torch.manual_seed(SEED)
symbols, to_id, training, validation = prepare(TEXT.read_text(encoding="utf-8"))
test_model = GPT(len(symbols))
print(len(symbols), "symbols ·", sum(p.numel() for p in test_model.parameters()), "parameters")
x, y = get_batch(training, torch.Generator().manual_seed(0))
print("Batch:", tuple(x.shape), "→ logits:", tuple(test_model(x).shape))

90 symbols · 215002 parameters
Batch: (32, 64) → logits: (32, 64, 90)


## 4. Evaluating and training

In [7]:
def mean_loss(model, x, y):
    logits = model(x)
    return F.cross_entropy(logits.view(-1, logits.shape[-1]), y.view(-1))


@torch.no_grad()
def evaluate(model, data, batches=20):
    generator = torch.Generator().manual_seed(0)
    losses = []
    for _ in range(batches):
        x, y = get_batch(data, generator)
        losses.append(mean_loss(model, x, y).item())
    return sum(losses) / len(losses)


def train(model, training, validation):
    optimizer = torch.optim.AdamW(model.parameters(), lr=RATE)
    generator = torch.Generator().manual_seed(SEED)
    history = []
    for step in range(STEPS + 1):
        if step % 250 == 0:
            measure = (step, evaluate(model, training),
                       evaluate(model, validation))
            history.append(measure)
            print("Step %4d · training %.3f · validation %.3f" % measure)
        if step == STEPS:
            break
        x, y = get_batch(training, generator)
        loss = mean_loss(model, x, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
    return history

## 5. Generating, checking the mask and running

The last cell trains the complete model: it takes about six minutes.

In [8]:
@torch.no_grad()
def generate(model, symbols, to_id, start, n, generator):
    ids = torch.tensor([[to_id[c] for c in start]], dtype=torch.long)
    for _ in range(n):
        logits = model(ids[:, -CONTEXT:])[:, -1, :]
        probabilities = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probabilities, 1, generator=generator)
        ids = torch.cat([ids, next_id], dim=1)
    return "".join(symbols[i] for i in ids[0].tolist())


@torch.no_grad()
def check_mask(model, data, position=CONTEXT // 2):
    x = data[:CONTEXT].unsqueeze(0)
    changed = x.clone()
    changed[0, position] = (x[0, position] + 1) % model.output.out_features
    before, after = model(x), model(changed)
    past_same = torch.allclose(before[0, :position], after[0, :position])
    rest_changes = not torch.allclose(before[0, position:],
                                      after[0, position:])
    return past_same, rest_changes


def bigram(training, validation, vocab_size):
    counts = torch.ones(vocab_size, vocab_size)
    ones = torch.ones(len(training) - 1)
    pairs = (training[:-1], training[1:])
    counts.index_put_(pairs, ones, accumulate=True)
    table = counts / counts.sum(dim=1, keepdim=True)
    return -table[validation[:-1], validation[1:]].log().mean().item()


def run():
    torch.set_num_threads(4)
    torch.manual_seed(SEED)
    text = TEXT.read_text(encoding="utf-8")
    symbols, to_id, training, validation = prepare(text)
    model = GPT(len(symbols))
    n_params = sum(p.numel() for p in model.parameters())
    print("Symbols:", len(symbols), "· parameters:", n_params)
    reference = bigram(training, validation, len(symbols))
    print("Character bigram, validation: %.3f" % reference)
    rng = torch.Generator().manual_seed(7)
    soup = generate(model, symbols, to_id, "In a village", 60, rng)
    print("Sample before training:", repr(soup))
    start = time.perf_counter()
    history = train(model, training, validation)
    print("Training: %.0f seconds" % (time.perf_counter() - start))
    mask = check_mask(model, validation)
    print("Causal mask (past unchanged, rest changes):", mask)
    rng = torch.Generator().manual_seed(7)
    print(generate(model, symbols, to_id, "In a village", 400, rng))
    return model, history

In [9]:
model, history = run()

Symbols: 90 · parameters: 215002
Character bigram, validation: 2.413
Sample before training: 'In a villageœZDYxàbrcbÁfsOH6p“10mH x?gùæ àNSœqWBAL_tJQr”ZÑTI&ZŒàPtlAD1Bk'
Step    0 · training 4.591 · validation 4.598
Step  250 · training 2.162 · validation 2.181
Step  500 · training 1.894 · validation 1.915
Step  750 · training 1.752 · validation 1.777
Step 1000 · training 1.678 · validation 1.709
Step 1250 · training 1.612 · validation 1.650
Step 1500 · training 1.577 · validation 1.610
Step 1750 · training 1.540 · validation 1.578
Step 2000 · training 1.524 · validation 1.557
Step 2250 · training 1.503 · validation 1.540
Step 2500 · training 1.488 · validation 1.528
Step 2750 · training 1.468 · validation 1.504
Step 3000 · training 1.457 · validation 1.498
Training: 337 seconds
Causal mask (past unchanged, rest changes): (True, True)
In a village watchered, servess mi it in the long high
he should to the rugs of himself to be garmerds, and able that wife in we pled that, betnly becood. 

## 6. Experiment

Generate more text with another start or change the seed of the sampling. For exercise 12, change the position in `check_mask`.

In [10]:
print(generate(model, symbols, to_id, "Sancho Panza", 300, torch.Generator().manual_seed(3)))
print(check_mask(model, validation, position=10))

Sancho Panzassallety that was the
fitty that trove; but and then founding my accestiful of readown
village houre for distinctionied to him, her husbanded, and the trified the
princess. Thene conbraryyateful wound to my room the irone abody, the forrable,
Bote this thing those of the pawny brought obsen. Ey or d
(True, True)


## Exercises

1. Write the table of weights of the average for a sequence of four positions. Propose a row of scores, with −∞ in the future, that produces the row of position 3.

2. Compute the dot product of [2,1] with [1,3] and with [-1,2]. Which of the two keys matches the query [2,1] better?

3. Add to the example a fourth position with query [1,1], key [1,-1] and value [3,1]. Follow the four steps of attention and compute its attention weights and its output.

4. Change the query of position 3 in the example to [0,1] and compute its new weights. Which positions does it mostly attend to now?

5. If the queries and keys had 4 components, we would have to divide by √4 = 2. Compute the softmax of [6, 0] and of [6/2, 0]. What effect does dividing by the square root have?

6. Explain what would happen during training and when generating text if we removed the causal mask.

7. Why is the position embedding needed? Give an example of two texts that, without it, attention could not tell apart.

8. Normalize the vector [1, 3, 5, 7] by hand: compute its mean, its standard deviation and the result.

9. Explain in your own words what advantage there is in adding the output of each block, x + f(x), instead of replacing the vector with f(x).

10. Count the parameters of one head, of one block and of the complete model if the vectors had 128 numbers instead of 64, with the same 4 heads and an MLP that is still four times wider (512).

11. Run the program and write down the final training and validation losses. Compute e^(−loss) for your model and for the bigram, and interpret the difference.

12. Checking the mask. Call `check_mask` changing the character at position 10, and then the one at position 60. Explain the results and what they prove. There is no need to train: the mask works the same way with a freshly created model, `GPT(90)`, so you can do it in the notebook in a few seconds.

13. Change `CONTEXT` to 16 and train again. Compare the validation loss with that of the 64-character model and explain the difference.

## References

Based on the lecture [Let's build GPT](https://www.youtube.com/watch?v=kCc8FmEb1nY) and the code [ng-video-lecture](https://github.com/karpathy/ng-video-lecture) by Andrej Karpathy. Text of *Don Quixote* in John Ormsby's translation, in the public domain. The explanations and the exercises are original.